# 00 — Setup e verificação do ambiente

Execute este notebook uma vez ao configurar o projeto em um computador novo.
Ele instala dependências, verifica GPU e prepara a estrutura de pastas.

**Pré-requisitos (antes deste notebook):**
- Conda env com Python **3.11 ou 3.12** já criado e ativo (ACE-Step 1.5 não suporta 3.13+).
  ```
  conda create -n music-ai python=3.11 pip -y
  conda activate music-ai
  ```
- VS Code apontando este notebook para o kernel do env `music-ai`.

**Tempo estimado:** 5–30 minutos (download dos modelos ACE-Step pode demorar).

## 1. Verificar versão do Python

ACE-Step 1.5 declara `requires-python = ">=3.11,<3.13"`. Versões fora dessa faixa não terão wheels compatíveis para várias dependências (torchcodec, vector-quantize-pytorch, etc.).

In [ ]:
import sys
print(f'Python: {sys.version}')
assert (3, 11) <= sys.version_info < (3, 13), \
    f'ACE-Step exige Python 3.11 ou 3.12 — você está em {sys.version_info.major}.{sys.version_info.minor}'

## 2. Instalar PyTorch com CUDA

ACE-Step 1.5 no Windows exige `torch==2.7.1+cu128`. Verifique a versão do driver CUDA do sistema com `nvidia-smi` (deve mostrar CUDA Version ≥ 12.8 para usar `cu128`).

Outras plataformas: ver `requirements.txt`.

In [ ]:
# Descomente APENAS a linha correspondente ao seu ambiente:

# Windows com NVIDIA CUDA 12.8 (versão oficial do ACE-Step 1.5):
# !pip install torch==2.7.1+cu128 torchvision==0.22.1+cu128 torchaudio==2.7.1+cu128 --index-url https://download.pytorch.org/whl/cu128

# Linux x86_64 com NVIDIA CUDA 12.8:
# !pip install torch==2.10.0+cu128 torchvision==0.25.0+cu128 torchaudio==2.10.0+cu128 --index-url https://download.pytorch.org/whl/cu128

# macOS Apple Silicon (CPU/MPS):
# !pip install torch torchaudio torchvision

# CPU only (qualquer plataforma — geração muito lenta):
# !pip install torch torchaudio --index-url https://download.pytorch.org/whl/cpu

print('Descomente a linha correta acima e execute novamente.')

## 3. Instalar demais dependências do projeto

Demucs, librosa, jupyter e utilitários. (PyTorch já foi instalado acima; ACE-Step vem no passo 4.)

In [ ]:
!pip install -r ../requirements.txt

## 4. Clonar e instalar ACE-Step 1.5

Clona o repo na raiz do projeto (`../ACE-Step-1.5`) e instala como pacote editável.  
Modelos (~5 GB) são baixados automaticamente na primeira execução do notebook 02.

In [ ]:
import os
from pathlib import Path

root = Path('..').resolve()
ace_dir = root / 'ACE-Step-1.5'

if not ace_dir.exists():
    print('Clonando ACE-Step 1.5...')
    rc = os.system(f'git clone https://github.com/ace-step/ACE-Step-1.5.git "{ace_dir}"')
    assert rc == 0, 'Falha ao clonar — verifique conexão / git'
else:
    print(f'ACE-Step já clonado em {ace_dir}')

print('Instalando ACE-Step como pacote editável (pode demorar)...')
rc = os.system(f'pip install -e "{ace_dir}"')
print('✓ Concluído.' if rc == 0 else '✗ Erro na instalação — veja log acima.')

## 5. Registrar kernel Jupyter para o env

Faz com que o env apareça no seletor de kernel do VS Code/Jupyter como `Python (music-ai)`.

In [ ]:
!python -m ipykernel install --user --name music-ai --display-name "Python (music-ai)"

## 6. Verificar GPU

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
from scripts.utils import verificar_gpu

device = verificar_gpu()

## 7. Verificação completa do ambiente

In [ ]:
%run ../scripts/check_env.py

## 8. Estrutura de pastas

Cria as pastas para áudios, modelos e logs (idempotente).

In [ ]:
from pathlib import Path

pastas = [
    '../audio/raw',
    '../audio/stems',
    '../audio/output',
    '../models/lora',
    '../docs',
]

for p in pastas:
    Path(p).mkdir(parents=True, exist_ok=True)
    print(f'  ✓ {p}')

print('\nColoque suas gravações originais (WAV) em: audio/raw/')

## Próximo passo

Coloque sua gravação de voz + violão (em WAV) na pasta `audio/raw/`  
e abra o notebook `01_separar_stems.ipynb`.